# 4차시 실습 — 저장하고, 배포하고, 예쁘게

**남는 데이터(localStorage→Supabase) + 공개 URL 2개(GitHub Pages·Vercel) + 디자인 업그레이드 (Codex 단독)**

## 🧪 이 노트북 사용법

- 이 노트북은 워크샵 **실습(Codex CLI)의 실행 가능한 동반 자료**입니다. 각자 발급받은 **`OPENAI_API_KEY`** 로 바로 돌아갑니다.
- 실제 캠프에서는 **터미널의 Codex CLI**로 같은 작업을 합니다. 그래서 각 단계마다
  - **▶︎ 노트북에서 실행** (파이썬 셀) 과
  - **⌨️ 터미널 Codex 명령** (복붙용) 을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 셀을 실행하세요 (`Shift+Enter`).
- 황금률: **작게 요청 → 확인 → 수정** (1차시 iteration loop).

> ⚠️ API 호출에는 약간의 토큰 비용이 듭니다. 기본 모델은 저렴한 `gpt-4o-mini`. **설치·키 발급·Codex 로그인은 0차시에서 끝낸 전제**입니다 — 안 됐으면 `0_setup/설치_가이드.md` 를 먼저 보세요.

In [ ]:
# ✅ 환경 준비 — 맨 처음 한 번만 실행
%pip install -q openai
import os, getpass, re, pathlib, json
from openai import OpenAI
from IPython.display import IFrame, Markdown, display

# 각자 발급받은 OPENAI_API_KEY 입력 (입력값은 화면에 보이지 않습니다)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY를 붙여넣고 Enter: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # 품질을 더 원하면 "gpt-4o" 또는 "gpt-4.1" 로 바꾸세요

def ask(prompt, system=None, **kw):
    """프롬프트 하나를 보내고 답변 텍스트를 돌려준다."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=msgs, **kw)
    return r.choices[0].message.content

def show(text):
    display(Markdown(text))

print("준비 완료! MODEL =", MODEL)

In [ ]:
# 🔧 HTML 생성·미리보기 도우미 (HTML을 만드는 차시에서 사용)
def extract_code(text):
    """답변에서 첫 번째 코드블록만 꺼낸다. 없으면 원문 그대로."""
    if "```" not in text:
        return text.strip()
    block = text.split("```", 2)[1]
    if "\n" in block:
        first, rest = block.split("\n", 1)
        if first.strip().isalpha():   # ```html 같은 언어 태그 제거
            return rest.strip()
    return block.strip()

def save_and_preview(html, filename="app.html", height=420):
    pathlib.Path(filename).write_text(html, encoding="utf-8")
    print("저장됨 →", filename, " (브라우저로 직접 열어도 됩니다)")
    return IFrame(filename, width="100%", height=height)

def iterate(current_html, instruction):
    """현재 HTML 전체를 보여주고(=컨텍스트) 한 가지 수정을 요청한다."""
    prompt = f"""아래는 현재 HTML 전체야:
```html
{current_html}
```
다음 한 가지만 반영해서 **전체 HTML을 다시** 출력해줘: {instruction}
- 설명 없이 HTML 코드만 출력해."""
    return extract_code(ask(prompt))

## 시작 — 앞 차시 앱 불러오기

`app.html`(3차시 또는 1차시)이 있으면 불러오고, 없으면 작은 샘플로 진행합니다.

In [ ]:
src = None
for f in ["app.html"]:
    if pathlib.Path(f).exists():
        src = f; break
if src:
    html = pathlib.Path(src).read_text(encoding="utf-8")
    print("불러옴 →", src)
else:
    html = "<!doctype html><html lang=ko><body><h1>메모</h1><input id=t><button onclick=\"add()\">추가</button><ul id=l></ul><script>function add(){let li=document.createElement('li');li.textContent=t.value;l.appendChild(li);t.value=''}</script></body></html>"
    print("앞 차시 파일 없음 — 샘플 메모 앱으로 진행")
save_and_preview(html, "app.html")

## STEP 1 — 데이터 영속화 ① localStorage 

새로고침해도 데이터가 남도록 **localStorage**에 저장합니다. 데모에는 이걸로 충분합니다 (가입·카드 0).

> 🧩 **용어:** **localStorage** = *브라우저 안에 있는 작은 메모장*. 입력한 값을 그 브라우저에 저장해 둬서, 창을 닫았다 열어도 남아 있습니다. (단, **그 컴퓨터·그 브라우저에만** 저장돼요.)

⌨️ **터미널 Codex:** `codex "새로고침해도 데이터가 남도록 localStorage 저장 기능을 추가해줘"`

In [ ]:
html = iterate(html, "새로고침해도 데이터가 남도록 localStorage에 저장하고, 페이지 로드 시 불러오게 해줘.")
save_and_preview(html, "app.html")

## STEP 1-B — 데이터 영속화 ②: **Supabase = 클라우드 DB** 

localStorage는 **내 브라우저에만** 저장됩니다 — 친구 폰에서 열면 빈 화면. 기기·사람 간 공유하려면 클라우드 DB가 필요합니다.
**Supabase** = 무료로 시작하는 관리형 Postgres(데이터베이스) + **자동 생성 REST API**. 서버 코드 없이 브라우저에서 바로 읽고 씁니다.

> 🧩 **비유:** localStorage가 *내 수첩*이라면, Supabase는 *인터넷에 있는 공용 스프레드시트*. 누가 어디서 열어도 같은 데이터가 보입니다.

**오늘의 흐름 — 콘솔 ①~④ → 노트북 테스트 ⑤ → 앱에 적용 ⑥:**

| 단계 | 어디서 | 하는 일 |
|---|---|---|
| ① 프로젝트 만들기 | supabase.com | 무료 가입 → New project |
| ② 테이블 만들기 | Table Editor | `memos` 표 만들기 (`text` 컬럼) |
| ③ 접근 설정 3가지 | Table Editor · Authentication · SQL Editor | 정책 2개 + 익명 로그인 + GRANT 3줄 ← **다들 여기서 막힘!** |
| ④ 열쇠 복사 | Settings → API Keys | Project URL + **publishable key** |
| ⑤ 연결 테스트 | 이 노트북 | 파이썬으로 저장·조회 성공 확인 |
| ⑥ 앱에 적용 | 이 노트북/Codex | localStorage → Supabase 교체 |

아래 셀들을 순서대로 따라 하세요.

### ① 프로젝트 만들기 & ② `memos` 테이블 만들기

**① 프로젝트 만들기 (supabase.com)**
1. [supabase.com](https://supabase.com) → **Start your project** → **Continue with GitHub** 로 가입 (카드 등록 불필요)
2. **New project** 클릭 → 항목 입력:
   - **Name**: `lunch-app` (아무거나)
   - **Database Password**: **Generate a password** 클릭 (오늘 다시 쓸 일 없음)
   - **Region**: **Northeast Asia (Seoul)** — 가까울수록 빠름
3. **Create new project** → 준비되는 데 **1~2분** (프로젝트 상태가 초록불이 될 때까지 대기)

**② 테이블 만들기 (왼쪽 메뉴 Table Editor)**
1. 왼쪽 사이드바 **Table Editor**(표 아이콘) → **Create a new table**
2. **Name**: `memos` — ⚠️ 앱 코드와 철자가 정확히 같아야 합니다
3. **Enable Row Level Security (RLS)** 체크는 **켠 상태 그대로** 둡니다 (다음 단계 ③에서 정책을 추가)
4. **Columns**: `id`, `created_at` 두 줄은 이미 있음 → **Add column** 으로 한 줄만 추가:
   - **Name**: `text` / **Type**: `text`
5. **Save**

> 🧩 **용어:** **테이블(table)** = 엑셀 시트 하나. **컬럼(column)** = 그 시트의 열 제목. 우리 표는 `id`(자동 번호) · `created_at`(자동 저장 시각) · `text`(메모 내용) 3열짜리입니다.

### ③ 접근 설정 3가지 — 정책 2개 · 익명 로그인 · GRANT (⚠️ 최대 관문)

**왜 필요한가:** publishable key는 브라우저에 노출되는 **공개용 열쇠**이고, 이 키의 요청은 **anon(익명) 롤**로 실행됩니다. 그래서 Supabase는 기본적으로 **모든 문을 잠가두고(RLS ON)**, 우리가 "이 표는 읽기 OK / 쓰기 OK"라고 **정책(Policy)** 으로 문을 열어줘야 합니다. 정책이 없으면 에러도 아닌 **빈 결과**만 돌아와서 한참 헤매게 됩니다 — 지금 바로 열어두세요.

> 🧩 **용어:** **RLS(Row Level Security, 행 수준 보안)** = 표의 줄(행) 단위로 "누가 읽고/쓰고/지울 수 있나"를 정하는 잠금장치. **Policy(정책)** = 그 잠금장치에 다는 허용 규칙 하나.

**정책 2개 — 대시보드 템플릿 그대로 (Table Editor에서 `memos` → Add RLS policy → Create policy):**
1. **읽기:** 템플릿 **"Enable read access for all users"** 선택 → 그대로 **Save policy** (APPLIED TO: `public`)
2. **쓰기:** 템플릿 **"Enable insert for authenticated users only"** 선택 → 그대로 **Save policy** (APPLIED TO: `authenticated`)

정책 목록에 위 2개가 보이면 절반 성공. 그런데 쓰기 정책이 **authenticated(로그인한 사용자) 전용**이죠 — 로그인 화면을 만들 필요는 없습니다. **익명 로그인**을 켜면 코드 한 줄로 통과됩니다:

3. **익명 로그인 허용:** 왼쪽 메뉴 **Authentication → Sign In / Providers** → **Allow anonymous sign-ins** 켜기 → Save
4. 코드에서는 `await db.auth.signInAnonymously()` 한 줄 (⑥에서 Codex가 넣어줍니다) — 익명 유저도 **authenticated 롤**을 받아 INSERT가 통과됩니다.

**마지막 하나 — 테이블 권한(GRANT) 1회:** 최근 만든 프로젝트는 보안 강화로 **테이블 권한이 기본 잠금**입니다(RLS 이전 단계 — 이게 없으면 정책이 있어도 403 `permission denied for table`). 왼쪽 메뉴 **SQL Editor** → New query → 아래 3줄 붙여넣고 **Run**:
```sql
grant usage  on schema public       to anon, authenticated;
grant select on table public.memos  to anon, authenticated;
grant insert on table public.memos  to authenticated;
```

> ⚠️ 익명 로그인은 "버튼만 누르면 누구나 쓰는" 데모용 구성입니다. 실제 서비스에선 익명 가입 남용 방지로 **CAPTCHA**를 붙이거나 정식 로그인으로 좁힙니다(오늘 범위 밖). 익명 유저의 JWT에는 `is_anonymous` 표시가 있어 정책에서 구분할 수도 있습니다.
> 콘솔 화면은 가끔 바뀝니다 — 버튼 이름이 조금 달라도 **"SELECT용 1개, INSERT용 1개 정책을 만든다"** 는 목표는 같습니다.

### ④ Project URL·publishable key 복사 + 동작 원리

1. 왼쪽 아래 **⚙ Settings → API Keys**
2. 두 값을 복사해 두세요 (다음 셀에 붙여넣습니다):
   - **Project URL**: `https://xxxx.supabase.co`
   - **Publishable key** (`sb_publishable_...`) — 브라우저에 노출돼도 되는 공개 키
   - (참고: 옛 자료의 **anon key**가 이 역할이었습니다 — 레거시 JWT 키는 **2026년 말 폐기 예정**이라 지금은 publishable key를 씁니다)
3. ⚠️ **Secret key(`sb_secret_...`, 구 `service_role`)는 절대 복사 금지** — 모든 잠금(RLS)을 무시하는 마스터키라 브라우저 코드에 넣으면 안 됩니다.

**동작 원리 — CDN 한 줄이면 정적 HTML에서 바로:**
```html
<script src="https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2"></script>
<script>
  const db = supabase.createClient("https://내프로젝트.supabase.co", "sb_publishable_...");
  await db.auth.signInAnonymously();                            // 익명 로그인 → authenticated 롤
  await db.from("memos").insert({ text: "점심: 김치찌개" });   // 저장 (INSERT 정책 통과)
  const { data } = await db.from("memos").select();            // 불러오기
</script>
```
설치·빌드 없음 — `<script>` 한 줄로 브라우저가 Supabase의 **자동 생성 REST API**에 직접 요청합니다. 서버는 Supabase가 대신 운영해 줍니다.

In [ ]:
# 🔌 ⑤ 연결 테스트 — 앱을 고치기 전에, 콘솔 설정(①~④)이 맞는지 파이썬으로 먼저 확인!
%pip install -q requests
import requests

SUPABASE_URL = "https://project.supabase.co"   # ← ④에서 복사한 Project URL
SUPABASE_KEY = "sb_puUx"             # ← ④에서 복사한 publishable key

# 0) 익명 로그인 — INSERT 정책(authenticated 전용)을 통과할 토큰 받기
r = requests.post(f"{SUPABASE_URL}/auth/v1/signup",
                  headers={"apikey": SUPABASE_KEY, "Content-Type": "application/json"}, json={})
token = r.json().get("access_token")
print("익명 로그인:", r.status_code, "✅ 성공" if token else "❌ 실패 → Authentication에서 Allow anonymous sign-ins 켜기")

headers = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {token or SUPABASE_KEY}"}
memos_api = f"{SUPABASE_URL}/rest/v1/memos"   # Supabase가 테이블마다 자동으로 만들어주는 REST 주소

# 1) 쓰기 테스트 (INSERT) — 201이면 성공
r = requests.post(memos_api, headers={**headers, "Content-Type": "application/json"},
                  json={"text": "노트북에서 보낸 테스트 메모"})
print("쓰기:", r.status_code, "✅ 성공" if r.status_code in (200, 201) else "❌ 실패 → " + r.text[:200])

# 2) 읽기 테스트 (SELECT) — 방금 쓴 메모가 보여야 성공
r = requests.get(memos_api + "?select=*", headers=headers)
rows = r.json() if r.ok else []
print("읽기:", r.status_code, f"→ {len(rows)}건", rows[-3:] if rows else "❌ 0건 (아래 🆘 표 참고)")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
익명 로그인: 200 ✅ 성공
쓰기: 201 ✅ 성공
읽기: 200 → 1건 [{'id': 1, 'created_at': '2026-07-06T17:35:07.642499+00:00', 'text': '노트북에서 보낸 테스트 메모'}]


### 🆘 Supabase 문제 해결

| 증상 | 원인 → 해결 |
|---|---|
| 쓰기/읽기 `403` + `permission denied for table` (GRANT 힌트 포함) | **테이블 GRANT 누락** — 새 프로젝트는 기본 잠금 → ③의 GRANT 3줄을 SQL Editor에서 실행 |
| 쓰기 `401 Invalid API key` | 키를 잘못 복사 → ④에서 **publishable** key(`sb_publishable_...`)를 다시 복사 (따옴표 안에 통째로 붙여넣기) |
| 쓰기 `404` / `relation "..." does not exist` | 테이블 이름 불일치 → 콘솔의 표 이름과 코드의 `memos` 철자 확인 |
| 쓰기 `violates row-level security policy` 에러 | **INSERT 정책 누락** 또는 **익명 로그인 실패**(토큰 없이 anon 롤로 요청됨) → ③의 정책 2개와 익명 로그인 상태 확인 |
| 읽기가 200인데 항상 `[]` (0건) | **SELECT 정책 누락** → ③에서 읽기 허용 정책 추가 (에러가 아니라 빈 결과로 옴 — 함정!) |
| 익명 로그인 4xx (`anonymous_provider_disabled`) | **Allow anonymous sign-ins 꺼짐** → ③에서 Authentication 설정 켜기 |
| `Failed to fetch` / 응답 없음 | URL 오타(`https://` 포함 전체인지) 또는 프로젝트 일시정지 → 콘솔에서 **Restore project** |
| 테스트 셀은 성공인데 앱에선 안 됨 | 생성된 `app.html`에 URL/key가 placeholder 그대로 → 파일 열어 실제 값이 들어갔는지 확인 |

> ✅ 테스트가 성공했다면 콘솔의 **Table Editor → memos** 를 열어보세요 — 방금 쓴 메모가 표에 그대로 보입니다. 이 화면이 여러분의 "관리자 페이지"입니다.

In [9]:
# ⑥ 앱에 적용 — localStorage 대신 Supabase에 저장 (위 ⑤ 테스트 셀의 변수를 그대로 사용)
html = iterate(html, f"""localStorage 대신 Supabase에 저장하도록 바꿔줘.
- <script src=\"https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2\"></script> 를 추가
- createClient("{SUPABASE_URL}", "{SUPABASE_KEY}") 로 클라이언트 생성
- 페이지 로드 시 await db.auth.signInAnonymously() 로 익명 로그인 먼저 (INSERT 정책 통과용)
- 테이블 memos(text 컬럼)에 insert, 로드 시 select로 불러오기""")
save_and_preview(html, "app.html")
# ✅ 진짜 확인: 시크릿 창(또는 다른 브라우저)에서 app.html 열기 → 데이터가 보이면 성공.
#    localStorage였다면 빈 화면이었을 것 — 이게 클라우드 DB의 차이!
# ⌨️ 터미널 Codex: codex "localStorage 대신 Supabase에 저장하게 바꿔줘. URL은 ○○, publishable key는 ○○, 테이블은 memos(text 컬럼)야. 로드 시 signInAnonymously로 익명 로그인 먼저 해줘"

저장됨 → app.html  (브라우저로 직접 열어도 됩니다)


## 잠깐 — **Git? GitHub? 그게 뭐예요?**

배포에 들어가기 전, 처음 듣는 단어들을 쉬운 말로 정리합니다. (외울 필요 없이 한 번만 읽어두세요.)

| 용어 | 한 줄 비유 | 풀어 말하면 |
|---|---|---|
| **Git** | 파일의 *타임머신/세이브 기능* | 내 컴퓨터에서 파일이 바뀐 이력을 단계별로 저장해두는 프로그램. 잘못돼도 이전으로 되돌릴 수 있어요. |
| **GitHub** | *코드용 구글드라이브 + 웹 호스팅* | Git으로 저장한 프로젝트를 **인터넷에 올려두는 사이트**. 백업·공유·공개가 됩니다. |
| **레포(Repository)** | *프로젝트 폴더 한 칸* | 앱 하나가 들어가는 GitHub 안의 방. 파일 + 변경 이력이 함께 담깁니다. |
| **GitHub Pages** | *내 폴더를 그대로 웹사이트로* | GitHub에 올린 `index.html`을 **무료 웹주소로 공개**해주는 기능. 오늘 이걸 씁니다. |

> 그림으로: **내 PC(파일) → (Git으로 저장) → GitHub(인터넷에 업로드) → GitHub Pages(웹사이트로 공개)**
> 즉, *내 컴퓨터에만 있던 앱*을 *전 세계가 볼 수 있는 주소*로 바꾸는 과정입니다.

## STEP 2 — 배포 ①: **GitHub Pages**로 인터넷에 공개 

우리 앱은 **정적 파일(HTML 1개)** 이라 **GitHub Pages**로 **가입·카드 없이 무료** 공개할 수 있습니다.
오늘은 **GitHub Pages와 Vercel 두 곳 모두**에 배포해서 차이를 비교합니다. 먼저 Pages부터.

> 🔑 가장 중요한 규칙: 대문 페이지 파일 이름은 반드시 **`index.html`**.

아래 셀로 `index.html` 을 만든 뒤, 이어지는 ①~⑤를 그대로 따라 하세요.

In [10]:
pathlib.Path("index.html").write_text(html, encoding="utf-8")
print("index.html 생성 완료 — 이 파일을 GitHub 레포에 올리면 공개됩니다.")

index.html 생성 완료 — 이 파일을 GitHub 레포에 올리면 공개됩니다.


### ① GitHub 레포 만들기
- [github.com/new](https://github.com/new) 접속
- **Repository name** 입력(예: `lunch-app`) → 공개 범위는 **Public** 선택
- 맨 아래 **Create repository** 클릭
> Public이어야 무료 Pages가 가장 단순하게 됩니다.

### ② 내 앱 폴더를 레포에 올리기 (터미널, **앱이 있는 폴더**에서)
```bash
git init
git add .                       # index.html 등 필요한 파일 모두
git commit -m "first deploy"
git branch -M main
git remote add origin https://github.com/<내아이디>/<레포이름>.git
git push -u origin main
```
- `<내아이디>`/`<레포이름>` 은 ①에서 만든 값으로 바꾸세요.
- 처음 `push` 때 GitHub 로그인이 필요합니다(브라우저 또는 토큰).

**위 명령어, 한 줄씩 무슨 뜻이냐면:**

| 명령어 | 무슨 뜻 |
|---|---|
| `git init` | 이 폴더를 **Git이 관리하도록 시작**(이력 저장 켜기). 폴더당 맨 처음 한 번만. |
| `git add .` | 바뀐 파일들을 **저장할 목록에 담기**(`.` = 이 폴더 전체). |
| `git commit -m "..."` | 담은 파일을 **하나의 저장 지점으로 기록**. `-m` 뒤 따옴표는 *저장 메모*(무엇을 했는지). |
| `git branch -M main` | 기본 작업 줄기 이름을 **`main`으로** 맞추기(GitHub 표준). |
| `git remote add origin <주소>` | **내 PC ↔ GitHub 레포를 연결**. `origin` = 그 GitHub 주소의 별명. |
| `git push -u origin main` | 기록한 내용을 **GitHub로 업로드**(올리기). 이후엔 `git push` 만으로 됩니다. |

> ⌨️ **외우기 부담되면 Codex에 맡기세요:** `codex "이 폴더를 git 레포로 초기화하고 main 브랜치로 커밋·푸시해줘"` — Codex가 위 명령을 대신 실행해 줍니다.

### ②-B ✨ **Codex로 git에 올려보기 (직접 해보기)**

명령어를 손으로 치는 대신, **Codex에게 말로 시켜** 똑같이 git에 반영해 봅니다. 이게 바이브 코딩 방식이에요.

**1.** 앱이 있는 폴더에서 Codex를 켜기:
```bash
cd 내앱폴더
codex
```

**2.** 대화창에 아래를 **하나씩** 입력 (①에서 GitHub 레포를 먼저 만들어 두세요):
```text
이 폴더를 git 저장소로 초기화하고, 모든 파일을 "first deploy" 메시지로 커밋해줘.
```
```text
원격 저장소 https://github.com/<내아이디>/<레포이름>.git 에 연결하고 main 브랜치로 push 해줘.
```

**3.** 나중에 앱을 고쳤다면, 다시 Codex에게:
```text
방금 바꾼 내용을 "update" 메시지로 커밋하고 push 해줘.
```

**무슨 일이 일어나나:**
- Codex가 `git init → add → commit → remote add → push` 명령을 **스스로 만들어 실행**합니다.
- 실행 전 **"이 명령을 실행할까요?" 승인**을 물어볼 수 있어요 → 내용을 보고 **승인(허용)**.
- 처음 push 때 **GitHub 로그인/토큰**을 요구하면 화면 안내를 따르세요.
- 끝나면 GitHub 레포 페이지를 새로고침 → 내 파일이 올라와 있으면 성공! ✅

> 💡 명령어를 몰라도, **"~을 git에 올려줘"** 처럼 말하면 Codex가 알아서 처리합니다. (2차시 GitHub 연동의 연장선)

### ②-C 🔄 **수정할 때마다 자동으로 commit·push 하기**

매번 시키지 않아도 바뀔 때마다 자동 저장·업로드 되게 하는 방법입니다.
> ⚠️ Codex CLI 자체에는 *자동 커밋* 기능이 아직 없습니다(공식 미지원). 아래 두 우회법을 씁니다.

**방법 A — AGENTS.md에 규칙 넣기 (터미널 흐름 유지)**
프로젝트 `AGENTS.md` 에 아래를 추가하면, Codex가 **한 작업이 끝날 때마다** 알아서 커밋·푸시합니다.
```markdown
## Git 규칙
- 코드를 수정하면 항상 git add → commit(무엇을 했는지 메시지) → push 까지 수행한다.
```
- 승인 질문을 줄이려면 자동 승인 모드로 실행: `codex --full-auto`
- 한계: "글자 바뀔 때마다"가 아니라 **작업(프롬프트) 단위**. 지시 기반이라 100% 보장은 아님.

**방법 B — GitDoc 확장 (저장할 때마다 자동, 가장 쉬움) ✅**
도구와 무관하게 **파일을 저장(Cmd/Ctrl+S)하는 순간 자동 commit + push** 됩니다.
1. VS Code 확장에서 **GitDoc** 설치
2. 명령 팔레트(`Cmd/Ctrl+Shift+P`) → **`GitDoc: Enable`**
3. 이후 저장할 때마다 자동 커밋 → 기본 자동 push (`GitDoc: Autopush` 설정)
- 장점: 명령어 0개 / 단점: 커밋이 자주 쌓여 히스토리가 지저분(데모·개인용엔 무방)

> 정리: **터미널 유지 → A**, **가장 간단 → B**. 둘 다 *처음 1회는* ①(레포)·②(원격 연결)가 되어 있어야 동작합니다.

### ③ Pages 켜기 (GitHub 웹에서)
1. 방금 만든 레포 페이지 → 상단 **Settings**
2. 왼쪽 메뉴 **Pages**
3. **Build and deployment → Source**: `Deploy from a branch`
4. **Branch**: `main` 선택 / 폴더 **`/ (root)`** → **Save**

### ④ 공개 주소 확인 (1~2분 뒤)
- 같은 Pages 화면 위쪽에 주소가 뜹니다:
  **`https://<내아이디>.github.io/<레포이름>/`**
- 휴대폰으로 열어 바로 데모! (전 세계 누구나 접속)

### ⑤ 수정하면 다시 배포 (반복)
파일을 고친 뒤 아래 3줄이면 끝 — 1~2분 후 자동 반영됩니다:
```bash
git add . && git commit -m "update" && git push
```

### 🆘 배포 문제 해결
| 증상 | 해결 |
|---|---|
| 사이트가 **404** | 파일명이 **`index.html`** 인지, Pages의 **Branch=main / 폴더=root** 설정이 맞는지 확인 |
| 빈 화면 / 옛 화면 | 1~2분 더 기다린 뒤 **강력 새로고침**(Cmd/Ctrl+Shift+R) |
| `push` 인증 실패 | 비밀번호 대신 **토큰**(GitHub Settings → Developer settings → Personal access tokens) 사용, 또는 `gh auth login` |
| Pages에서 **Branch 선택지 없음** | 먼저 **main 브랜치로 1회 push** 되어 있어야 목록에 나타남 |
| 이미지/CSS가 안 보임 | 파일 경로를 **상대경로**로(`./style.css`), 대소문자 정확히 |

## STEP 3 — 배포 ②: **Vercel**로도 배포

**Vercel** = 폴더(또는 GitHub 레포)를 던지면 URL이 나오는 배포 서비스. 결과물은 Pages와 같지만 성격이 다릅니다:

| | GitHub Pages | Vercel |
|---|---|---|
| 반영 속도 | 1~2분 | **수 초~수십 초** |
| 재배포 | push 후 자동 (느긋) | push 후 **자동·즉시** |
| 주소 | `아이디.github.io/레포` | `프로젝트.vercel.app` |
| 확장성 | 정적 파일만 | 나중에 **서버 기능(API)·환경변수**까지 |

배포 방법 3가지 — 오늘은 **③ GitHub 연동**(아까 Pages에 쓴 레포 재활용)으로 갑니다:
1. **웹 드래그&드롭** — 가장 빠른 1회성 (아래 '대안' 참고)
2. **CLI** — 터미널파, Codex에 맡기기 좋음 (아래 '대안' 참고)
3. **GitHub 연동 ✅ 오늘의 방법** — 한 번 연결하면 이후 **`git push`만 해도 자동 재배포**

> 🧩 **왜 같은 앱을 두 곳에?** 배포 서비스마다 성격이 다르다는 걸 몸으로 비교해 보려는 것. 실전에서는 하나만 골라도 됩니다.

### GitHub 연동 배포 ①~⑥ (오늘의 방법, 5분)

1. **가입**: [vercel.com](https://vercel.com) → **Sign Up** → **Continue with GitHub** (Pages 때 쓴 GitHub 계정 그대로) → 플랜은 **Hobby**(무료) 선택
2. 대시보드 오른쪽 위 **Add New… → Project**
3. **Import Git Repository** 목록에서 아까 만든 레포(예: `lunch-app`) 옆 **Import** 클릭
   - 목록에 안 보이면: **Adjust GitHub App Permissions** → GitHub 화면에서 해당 레포(또는 All repositories) 접근 허용 → **Install/Save**
4. 설정 화면은 **그대로 두고** (Framework Preset: **Other**, Root Directory: `./`) → **Deploy**
5. 수십 초 뒤 🎉 완료 화면 → 미리보기 이미지 클릭(또는 **Continue to Dashboard** → **Visit**)
   → 내 주소: **`https://<프로젝트>.vercel.app`**
6. **자동 재배포 체험** — 이게 Vercel의 매력:
   ```bash
   # 앱을 아무거나 한 줄 고친 뒤 (예: 제목 텍스트)
   git add . && git commit -m "vercel test" && git push
   ```
   → Vercel 대시보드 **Deployments** 탭에 새 배포가 **자동 생성** → 수십 초 후 `vercel.app` 새로고침하면 반영 ✅

> ✅ 이제 공개 URL이 **2개**: `github.io`(Pages) + `vercel.app`(Vercel). 레포는 하나 — **`git push` 한 번이면 둘 다 갱신**됩니다.

### 대안 2가지 — 드래그&드롭 / CLI

**대안 A — 웹 드래그&드롭 (레포 없이 30초, 1회성)**
1. vercel.com 대시보드 → **Add New… → Project**
2. 페이지의 업로드 영역에 **앱 폴더를 통째로** 끌어다 놓기 → **Deploy**
- 장점: git 몰라도 됨 / 단점: 수정할 때마다 다시 끌어다 놓아야 함 (자동 재배포 X)

**대안 B — CLI (터미널파, Codex에 맡기기 좋음)**
```bash
npm i -g vercel        # 최초 1회 설치 (Node는 0차시에 설치됨)
cd 내앱폴더
vercel login           # 이메일 입력 → 받은 메일에서 확인 버튼 클릭
vercel                 # 질문은 전부 Enter(기본값) → 미리보기 URL이 나옴
vercel --prod          # 진짜 주소(프로덕션)로 배포
```

⌨️ **터미널 Codex:** `codex "이 폴더를 vercel CLI로 배포해줘. vercel이 없으면 설치부터 하고, 로그인이 필요하면 절차를 알려줘"`
- `vercel` 명령도 Codex가 대신 실행해 줍니다. 첫 로그인(이메일 인증)만 사람 몫.

In [ ]:
# 🌐 배포 확인 — 두 공개 URL이 실제로 살아있는지 노트북에서 점검
%pip install -q requests
import requests

MY_URLS = [
    "https://<내아이디>.github.io/<레포이름>/",   # ← STEP 2 Pages 주소로 교체
    "https://<프로젝트>.vercel.app",               # ← STEP 3 Vercel 주소로 교체
]
for url in MY_URLS:
    try:
        r = requests.get(url, timeout=10)
        ok = r.ok and "<html" in r.text.lower()
        print("✅" if ok else "❌", r.status_code, url)
    except Exception as e:
        print("❌", url, "→", e)
# 둘 다 ✅ 면 휴대폰으로도 열어보세요 — 진짜 '전 세계 공개'입니다.

### 🆘 Vercel 문제 해결

| 증상 | 원인 → 해결 |
|---|---|
| **404: NOT_FOUND** | 레포 최상위에 `index.html`이 없음 → 파일 위치 확인 (하위 폴더에 있다면 프로젝트 **Settings → Root Directory** 를 그 폴더로 지정) |
| Import 목록에 내 레포가 없음 | GitHub App 권한 문제 → **Adjust GitHub App Permissions** 클릭 → 해당 레포(또는 All repositories) 접근 허용 |
| push 했는데 반영 안 됨 | 대시보드 **Deployments** 탭 확인 — 새 배포가 **없으면** push한 브랜치가 `main`인지 확인, **있으면** 강력 새로고침(Cmd/Ctrl+Shift+R) |
| 배포는 됐는데 Supabase 데이터가 안 보임 | 브라우저 개발자도구(F12) → Console 에러 확인 — URL/publishable key가 placeholder(`내프로젝트`) 그대로인 경우가 대부분 |
| 로그인이 자꾸 꼬임 | Pages 때 쓴 것과 **같은 GitHub 계정**으로 "Continue with GitHub" 했는지 확인 |

## STEP 4 — 디자인 업그레이드 

그냥 시키면 AI는 어디서 본 듯한 화면(흔한 폰트·보라 그라데이션·카드 나열)을 만듭니다. 세 가지로 탈출합니다.

**방법 ① 디자인 결정을 프롬프트에 담기** — 색·폰트·레이아웃을 *내가 결정*해서 지시 (아래 셀)

**방법 ② frontend 디자인 스킬 설치** — OpenAI 공식 skills 카탈로그([github.com/openai/skills](https://github.com/openai/skills))의 `frontend` 스킬. 2차시에서 배운 스킬 설치 절차 그대로. 설치 후엔 서체 2개·포인트색 1개 제한, 카드 남발 금지, 코드 전에 비주얼 컨셉부터 쓰게 강제됩니다.
⌨️ **터미널 Codex:** 스킬 설치 후 → `codex "frontend 스킬을 적용해서 이 앱의 디자인을 다시 잡아줘. 기능은 그대로."`

**방법 ③ 3D 배경 한 스푼 — Vanta.js** — three.js 기반 움직이는 3D 배경을 `<script>` 2줄로 (효과 미리보기: [vantajs.com](https://www.vantajs.com/))
```html
<div id="vanta-bg" style="position:fixed;inset:0;z-index:-1"></div>   <!-- body 맨 앞 -->
...콘텐츠(배경은 투명하게)...
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r134/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/vanta/dist/vanta.waves.min.js"></script>
<script>VANTA.WAVES({ el: "#vanta-bg" })</script>   <!-- 셋 다 </body> 직전, 이 순서 -->
```
> ⚠️ **안 보이는 3대 원인:** init 스크립트가 `<head>`에 있음(요소 생성 전 실행) / `#vanta-bg` 요소 없음 / 불투명 배경이 캔버스를 덮음. 그래서 아래 셀 프롬프트는 이 구조를 **정확히 지시**합니다.

> **절제 3원칙:** wow 요소는 한 곳만 · 서체 2개/포인트색 1개 · 움직임은 은은하게.
> 다 됐으면 **재배포**: `git add . && git commit -m "design" && git push` → Pages·Vercel 둘 다 자동 갱신.

In [ ]:
# 방법 ① — 디자인 결정을 담은 프롬프트
html = iterate(html, "다크 테마로 바꾸고, 포인트 색은 #FF6B35 하나만 써. 제목은 큰 세리프, 본문은 산세리프. 카드 나열 대신 여백과 큰 타이포그래피로 위계를 잡아줘. 기능은 그대로.")
save_and_preview(html, "app.html")

In [6]:
# 방법 ③ — Vanta.js 3D 배경 (구조를 정확히 지시해야 확실히 나온다)
html = iterate(html, """Vanta.js WAVES 3D 배경을 추가해줘. 구조는 정확히 이렇게:
- <body> 맨 앞에 <div id="vanta-bg" style="position:fixed;inset:0;z-index:-1"></div> 추가
- </body> 직전에 이 순서로 스크립트 3개 추가:
  <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r134/three.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/vanta/dist/vanta.waves.min.js"></script>
  <script>VANTA.WAVES({ el: "#vanta-bg" })</script>
- body와 콘텐츠 컨테이너의 배경은 투명하게(3D 배경이 비쳐 보이게), 글자는 밝은 색으로 가독성 유지
- 기능과 기존 스크립트는 그대로""")
save_and_preview(html, "app.html")

# 자동 점검 — 3대 실패 원인 체크
ok_el = 'id="vanta-bg"' in html
ok_order = html.find("three.min.js") < html.find("vanta.waves") < html.find("VANTA.WAVES")
ok_late = html.find('id="vanta-bg"') < html.find("VANTA.WAVES")
print("✅ 요소" if ok_el else "❌ #vanta-bg 요소 없음 — 다시 요청",
      "| ✅ 순서" if ok_order else "| ❌ 스크립트 순서 어긋남",
      "| ✅ 위치" if ok_late else "| ❌ init이 요소보다 앞 — </body> 직전으로")
# 다른 효과: vantajs.com에서 골라 WAVES 자리만 바꿔 요청 (BIRDS·CLOUDS·NET 등)

저장됨 → app.html  (브라우저로 직접 열어도 됩니다)
✅ 요소 | ✅ 순서 | ✅ 위치


## ✅ 마무리 체크리스트

- [ ] localStorage로 새로고침 후에도 데이터 유지
- [ ] Supabase **연결 테스트 셀** 통과 (쓰기 ✅ · 읽기 ✅) — Table Editor에서 내 데이터 확인
- [ ] Supabase 테이블에 저장 — **다른 브라우저에서도 데이터 보임**
- [ ] Codex로 git 초기화·커밋·push 해봄
- [ ] 공개 URL 2개: GitHub Pages + Vercel — **배포 확인 셀** ✅ + 휴대폰 접속 확인
- [ ] `git push` → Vercel **자동 재배포** 확인 (Deployments 탭)
- [ ] 디자인 업그레이드 1가지 이상 (프롬프트/frontend 스킬/3D 배경)
- [ ] 수정 후 재배포(`git add . && git commit -m "update" && git push`)까지 한 바퀴